# Stacking

This notebook implements a stacking ensemble learning technique (meta-learning) for fake news detection.

## Optimization Problem

MIN Z

$$
Z = - \frac{1}{N} \displaystyle\sum_{i=1}^{N} α_{c_i} [w_1 y_i log(F(x_i)) + w_0 (1 - y_i) log(1 - F(x_i))] + λ Ω(θ)
$$

<br>SUBJECT TO <br><br>

$C_1$: Stacking meta-learner

$$F(x) = g(f_1(x), f_2(x), ⋯ , f_M(x))$$

$C_2$: Feature Mapping

$$x_i = ϕ (title_i, text_i, category_i, dataset_i)$$

$C_3$: Binary Labels

$$y_i ∈ \{0, 1\}$$

$C_4$: Category Weight

$$α_{c} = \frac{1}{freq(c_i)}$$

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$


## Colab Configuration

**Project Environment** : Run the code cell below to auto-detect data ingestion script. <br>
**Stand-alone Environment** : Skip the code cell below and upload the 'ingest_data.py' script from github.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/project/COMP3608PROJECT')

Mounted at /content/drive


## Environment Configuration

The following code cell contains dependencies like third-party libraries that will be used in this notebook.

In [5]:
from ingest_data import load_datasets
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp


## Data Ingestion

Load all fake news datasets using the shared 'ingest_data.py' script. <br>

The resulting dataframe `df` has five columns:
- `title` : title of news
- `text` : actual text content of news
- `category` : type of news
- `label` : 0 = fake, 1 = real
- `dataset` : source dataset of the news record

Duplicate and null text rows are removed during ingestion resulting in ~99K records across three datasets and eight original columns.

In [3]:
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...


100%|██████████| 18.1M/18.1M [00:01<00:00, 18.1MB/s]

Extracting zip of true.csv...


[bhavik] Loaded 'true.csv': 21417 rows


100%|██████████| 22.9M/22.9M [00:01<00:00, 20.6MB/s]

...

100%|██████████| 32.8M/32.8M [00:01<00:00, 20.9MB/s]


[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...


100%|██████████| 1.08M/1.08M [00:00<00:00, 2.12MB/s]

Extracting zip of real.csv...


[shawky] Loaded 'real.csv': 21869 rows
Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'fake.csv': 19999 rows

Dropped 648 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,468
Fake (0): 47,089
Real (1): 52,379

Rows per source:
shawkyelgendy          40,824
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,691
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Technology             2,882
  Business               2,849
  Science                2,789
------------------------------------------------------------


## Text Cleaning and Preprocessing Pipeline

Raw news text contains noise that degrades quality of TF-IDF. <br>

We concatenate `title` and `text` into a single `content` field, then apply a cleaning pipeline that:
- Converts all text to lower case
- Removes URLs
- Strips HTML entities and HTML tags
- Removes punctuation and digits
- Collapses whitespace

Note: Stemming/Lemmatisation is NOT applied at this stage. TF-IDF witha sublinear term-frequency scale handles term frequency inflation naturally <br>

The cleaned text is stored in the new column `content`.

In [4]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'https?//S+|www\.\S+', ' ', text) # Remove URLs
    text = re.sub(r'<[^>]+>', ' ', text) # Strip HTML Tags
    text = re.sub(r'&[a-z]+;', ' ', text) # Strip HTML Entities
    text = re.sub(r'[^a-z\s]', ' ', text) # Keep letters only
    text = re.sub(r'\s+', ' ', text).strip() # Collapse whitespace
    return text

df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).apply(clean_text)

df.head()


,title,text,label,category,dataset,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara,as u s budget fight looms republicans flip the...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara,u s military to accept transgender recruits on...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara,senior u s republican senator let mr mueller d...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara,fbi russia probe helped by australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara,trump wants postal service to charge much more...


## TF-IDF Feature Extraction

In [6]:
TFIDF_PARAMS = {
    'sublinear_tf': True,
    'max_features': 100000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'strip_accents': 'unicode',
    'analyzer': 'word',
    'token_pattern': r'\b[a-z]{2,}\b' # min 2 characters per word
}

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

print("TF-IDF Vectorizer Configured.")
print(f"    ngram_range  : {tfidf.ngram_range}")
print(f"    max_features : {tfidf.max_features:,}")
print(f"    sublinear_tf : {tfidf.sublinear_tf}")

TF-IDF Vectorizer Configured.
    ngram_range  : (1, 2)
    max_features : 100,000
    sublinear_tf : True
